# WILDA Team 6 - Stage 2-2: Clustering Analysis

**Project:** Customer Churn Analysis for Telecommunication Company

This notebook performs customer segmentation using K-Means clustering.

### Important modelling decision
`Churn` is **not** used as an input feature because K-Means is an unsupervised method. Churn is examined only after the clusters are created to help interpret the segments.

### Workflow
1. Prepare clustering features
2. Standardise continuous numerical variables
3. Use the elbow method to choose K
4. Train K-Means
5. Profile and label clusters
6. Visualise the results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

df = pd.read_csv("Dataset_ATS_v2.csv")
print(df.shape)
df.head()

## 1. Prepare clustering features

Nominal categorical variables are one-hot encoded. `tenure` and `MonthlyCharges` are standardised because K-Means is distance-based. `SeniorCitizen` remains a binary 0/1 indicator.

`Churn` is deliberately excluded from the feature matrix.

In [ ]:
categorical_cols = ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract']
continuous_cols = ['tenure', 'MonthlyCharges']

X_cat = pd.get_dummies(df[categorical_cols], dtype=float)

X_num = df[["SeniorCitizen", "tenure", "MonthlyCharges"]].astype(float).copy()
scaler = StandardScaler()
X_num[continuous_cols] = scaler.fit_transform(X_num[continuous_cols])

X = pd.concat([X_cat.reset_index(drop=True), X_num.reset_index(drop=True)], axis=1)

assert "Churn" not in X.columns
assert X.isna().sum().sum() == 0

print("Clustering matrix shape:", X.shape)
print(X.columns.tolist())

## 2. Determine the optimal number of clusters

The assignment requires the **elbow method**. K values from 2 to 10 are compared using Within-Cluster Sum of Squares (WCSS). The largest change in the slope of the WCSS curve occurs at **K = 3**, which is selected for the final model.

Silhouette scores are also calculated as a supporting diagnostic, not as the primary selection rule.

In [ ]:
ks = list(range(2, 11))
inertias = []
silhouettes = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X, labels))

second_diff = np.diff(np.array(inertias), n=2)
optimal_k = ks[int(np.argmax(second_diff)) + 1]

print("Elbow-selected K:", optimal_k)

plt.plot(ks, inertias, marker="o")
plt.axvline(optimal_k, linestyle="--")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

## 3. Train the K-Means model

A final K-Means model is trained with `K = 3`, `random_state = 42`, and `n_init = 20` for stable, reproducible results.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X)

print(df["Cluster"].value_counts().sort_index())

## 4. Cluster profiling

The clusters are interpreted using the original customer variables. Churn is included here **only as an observed outcome for interpretation**, not as a clustering input.

In [ ]:
profile = df.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    Avg_Tenure=("tenure", "mean"),
    Avg_MonthlyCharges=("MonthlyCharges", "mean"),
    SeniorCitizen_Rate=("SeniorCitizen", "mean"),
    Churn_Rate=("Churn", lambda s: (s == "Yes").mean()),
    FiberOptic_Rate=("InternetService", lambda s: (s == "Fiber optic").mean()),
    MonthToMonth_Rate=("Contract", lambda s: (s == "Month-to-month").mean()),
).round(3)

profile

## 5. Final segment labels

The labels are based on the measured cluster profiles rather than being decided in advance.

- **Cluster 0:** Lower-Cost Lower-Churn Customers
- **Cluster 1:** Newer Higher-Churn Customers
- **Cluster 2:** Established Higher-Cost Lower-Churn Customers

These labels make the segments easier to communicate to business stakeholders. `Churn` is used only for interpretation after the clusters have already been created.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

for c in sorted(df["Cluster"].unique()):
    mask = df["Cluster"] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], s=12, alpha=0.5, label=f"Cluster {c}")

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Customer Segments - PCA View")
plt.legend()
plt.show()

## Conclusion

The elbow analysis identified three customer segments. The final K-Means model was trained on the complete feature space after appropriate encoding and scaling. Cluster profiles, visualisations, and labels were then created to support business interpretation and churn analysis.